# National gauge-value survey (Kaggle)

Runs the multigauge study on about **90 USGS basins across the 18 hydrologic regions of the conterminous U.S.**
(5 per region) and maps where NOAA rain gauges improve the pre-trained AI streamflow model.

Each basin: public data are downloaded (USGS streamflow, Daymet, NOAA GHCN-Daily gauges); the CAMELS-US LSTM is
fine-tuned with and without the gauges (2 seeds); the gain in test NSE (Oct 2017 to Sep 2023) is recorded with its
seed-to-seed spread.

**Setup:** Settings → *Accelerator: GPU T4 x2 or P100* and *Internet: on*. Then **Save Version → Save & Run All (Commit)**.
The run stops by itself after about 10.5 hours. **To continue**, add this notebook's output as an input
(*Add Input → Your Work*) and commit again: finished basins are skipped.

Data: USGS, Daymet, NOAA GHCN-Daily; basin list from CAMELS (Newman et al., 2015; Addor et al., 2017; Zenodo record 15529996).

In [ ]:
import torch, shutil
from pathlib import Path
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
WORK = Path("/kaggle/working/national"); WORK.mkdir(parents=True, exist_ok=True)
!rm -rf /kaggle/temp/multigauge-hydro && git clone -q https://github.com/ogunbajoadeyinka/multigauge-hydro.git /kaggle/temp/multigauge-hydro
!pip install -q -e "/kaggle/temp/multigauge-hydro[download]"
import sys; sys.path.insert(0, "/kaggle/temp/multigauge-hydro/src")
REPO = Path("/kaggle/temp/multigauge-hydro")

## 1. Resume: bring in basins finished by an earlier run (if its output is attached)

In [ ]:
prev = [p.parent for p in Path("/kaggle/input").rglob("national_summary.csv")] if Path("/kaggle/input").exists() else []
for p in prev:
    for f in p.iterdir():
        dst = WORK / f.name
        if f.is_dir() and not dst.exists():
            shutil.copytree(f, dst)
        elif f.name == "failed.csv" and not dst.exists():
            shutil.copy(f, dst)
print("basins already finished:", len(list(WORK.glob("*/results/run_info.json"))))

## 2. Basin list: CAMELS gauges of 150–5,000 km², interleaved across the 18 regions

In [ ]:
import subprocess, numpy as np, pandas as pd
import multigauge as mg
from multigauge.national import read_camels_catalog, candidate_order, run_national, light_config, fig_national_map, HUC2_NAMES
BASE = "https://zenodo.org/api/records/15529996/files"
for n in ("camels_name.txt", "camels_topo.txt"):
    if not Path(f"/kaggle/temp/{n}").exists():
        subprocess.run(["wget", "-nv", "-O", f"/kaggle/temp/{n}", f"{BASE}/{n}/content"], check=True)
catalog = read_camels_catalog("/kaggle/temp/camels_name.txt", "/kaggle/temp/camels_topo.txt")
trained = set(open(REPO / "notebooks" / "camels_531_basins.txt").read().split())
catalog["in_pretraining_set"] = catalog.index.isin(trained)
order = candidate_order(catalog, per_region=5)
# fill-in pass for regions still short of basins: up to 10,000 km2, wider gauge search (see run_national)
fill_order = candidate_order(catalog, per_region=5, max_area_km2=10000)
print(len(catalog), "CAMELS basins;", len(order), "eligible;", order["huc_02"].nunique(), "regions")

## 3. Run (about 5 minutes per basin on a GPU; stops after 10.5 h, re-run to continue)

Regions with fewer than 5 basins after the standard pass are retried with a wider gauge search (25 km around the basin, shorter records allowed, basins up to 10,000 km²). Each basin records the search radius it used.

In [ ]:
from multigauge.nn import StreamflowModel
model = mg.load_pretrained("camels-us")
summary = run_national(order, WORK, model, per_region=5, time_budget_h=10.5, config=light_config(n_seeds=2),
                       fill_candidates=fill_order)
print(len(summary), "basins finished")

## 4. Results

In [ ]:
summary = summary.merge(catalog[["in_pretraining_set"]], left_on="site", right_index=True, how="left")
summary.to_csv(WORK / "national_summary.csv", index=False)
print(summary["verdict"].value_counts().to_string(), "\n")
print("median NSE without gauges:", round(float(summary["NSE_finetuned"].median()), 3),
      "| median gain:", round(float(summary["NSE_gain"].median()), 3))
print(summary.groupby("region")[["NSE_finetuned", "NSE_gain"]].median().round(3).to_string())
fig = fig_national_map(summary); fig.write_html(WORK / "national_map.html"); fig.show()

## 5. Download the results (small files only)

Download `national_results.zip` from the **Output** tab.

In [ ]:
!cd /kaggle/working && rm -f national_results.zip && zip -qr national_results.zip national -x "*report.html" "*predictions.csv" "*forcings.csv" "*rainfall.csv" "*streamflow.csv" "*checkpoint*" && ls -la national_results.zip